In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests 
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
#1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.

def get_json(url: str) -> dict: 
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

In [0]:
# Странице. Написати функцију fetch_all(first_url: str) -> list која прати next линкове и враћа записе са свих страна у једној листи

def fetch_all(first_url: str) -> list:
    records = []
    url = first_url


    while url :
        page = get_json(url)
        records.extend(page['data'])


        if page ['next']:
            url = urljoin(url, page['next'])
        else:
            url = None

    return records

In [0]:
orders = fetch_all(BASE_URL + 'orders/page-1.json')
len(orders)

1852

In [0]:
#3.Чување сирових података. Написати функцију save_json(records: list, path: str) ->None која листу речника уписује у JSON фајл без икаквих измена.

def save_json(records: list, path: str) -> None:
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii= False, indent=2)
        


In [0]:
# 4.Преузимање фајла. Функција је дата: копирати је без измена у своју ћелију и користити узадатку 5

def download_file(url: str, path: str) -> int:
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    with open(path, 'wb') as f:
        f.write(response.content)

    return len(response.content)

In [0]:
download_file(FILES_URL + 'teams.csv' , os.path.join(RAW_DIR, 'teams.csv'))

801

In [0]:
#5.Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у

def extract(raw_dir: str) -> dict:
    orders = fetch_all(BASE_URL + 'orders/page-1.json')

    save_json(orders, os.path.join(raw_dir, 'orders.json'))

    download_file(FILES_URL + 'products.csv', os.path.join(raw_dir, 'products.csv'))
    download_file(FILES_URL + 'teams.csv', os.path.join(raw_dir, 'teams.csv'))
    download_file(FILES_URL + 'matches.parquet', os.path.join(raw_dir, 'matches.parquet'))

    orders_df = pd.json_normalize(orders, sep='_')
    orders_df = orders_df.rename(columns={
        'customer_name': 'customer',
        'customer_city': 'city'
    })

    products_df = pd.read_csv(os.path.join(raw_dir, 'products.csv'))
    teams_df = pd.read_csv(os.path.join(raw_dir, 'teams.csv'))
    matches_df = pd.read_parquet(os.path.join(raw_dir, 'matches.parquet'))

    return {
        'orders': orders_df,
        'products': products_df,
        'teams': teams_df,
        'matches': matches_df
    }

In [0]:
data = extract(RAW_DIR)

print(data['orders'].shape)
print(data['products'].shape)
print(data['teams'].shape)
print(data['matches'].shape)

(1852, 8)
(108, 7)
(20, 5)
(120, 9)


In [0]:
# 6.Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame која чисти каталог производа

def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates(subset='product_id').copy()

    df['team_code'] = df['team_code'].fillna('').str.strip().str.upper()
    df['category'] = df['category'].str.strip().str.lower()

    df['price'] = df['price'].str.replace(' RSD', '', regex=False)
    df['price'] = df['price'].str.replace(',', '.', regex=False)
    df['price'] = pd.to_numeric(df['price'])

    df['active'] = df['active'].str.strip().str.lower() == 'yes'

    return df


In [0]:
products= clean_products(data['products'])

print(len(products))
print(products['price'].dtype)
print(products['active'].sum())

105
float64
101


In [0]:
# 7.Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame

def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['order_id'] = df['order_id'].str.strip().str.upper()
    df['customer'] = df['customer'].str.strip().str.title()
    df['city'] = df['city'].fillna('Unknown').str.strip().str.title()
    df['payment'] =df['payment'].str.strip()

    return df



In [0]:
# Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple која враћа(unique, duplicates) поorder_id , задржава први

def remove_duplicate_orders(df: pd.DataFrame) ->tuple:
    unique = df.drop_duplicates(subset='order_id')
    duplicates = df[df.duplicated(subset='order_id')]

    return unique, duplicates

In [0]:
orders_clean = clean_orders(data['orders'])

unique, duplicates = remove_duplicate_orders(orders_clean)


print(len(unique))
print(len(duplicates))

1840
12


In [0]:
# 9.Написати функцију validate_order(row: dict, product_ids: set) -> strkоја враћа'OK'за исправан ред, у супротном разлог

def validate_order(row: dict, product_ids: set) -> str:
    if not str(row['order_id']).startswith('NSS-'):
        return 'invalid_order'

    if pd.isna(row.get('quantity')):
        return 'missing_quantity'

    q = pd.to_numeric(row.get('quantity'), errors='coerce')

    if pd.isna(q) or q <= 0 or q % 1 != 0:
        return 'invalid_quantity'

    p = pd.to_numeric(row.get('unit_price'), errors='coerce')

    if pd.isna(p) or p <= 0:
        return 'invalid_price'

    if row['product_id'] not in product_ids:
        return 'unknown_product'

    return 'OK'


In [0]:
#10.
#Написати функцију transform_orders(df: pd.DataFrame, products: pd.DataFrame) ->pd.DataFrame

def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['quantity'] = df['quantity'].astype(int)
    df['unit_price'] = df['unit_price'].astype(float)
    df['order_date'] = pd.to_datetime(df['order_date'])

    df['total'] = df['quantity'] * df['unit_price']

    df = df.merge(
        products[['product_id', 'product_name', 'team_code', 'category']],
        on='product_id',
        how='left'
    )

    return df



In [ ]:

data = extract(RAW_DIR)

products = clean_products(data['products'])
orders = clean_orders(data['orders'])

unique, duplicates = remove_duplicate_orders(orders)

product_ids = set(products['product_id'])

valid_rows = []

for i, row in unique.iterrows():
    if validate_order(row.to_dict(), product_ids) == 'OK':
        valid_rows.append(i)

valid_orders = unique.loc[valid_rows]

rezultat = transform_orders(valid_orders, products)

print(rezultat.shape)

In [ ]:
#11.Написати функцију split_orders(df: pd.DataFrame, products: pd.DataFrame) ->tuple која сирове поруџбине чисти, избацује дупликате, проверава и враћа торку (valid,rejected)


def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    orders = clean_orders(df)

    unique, duplicates = remove_duplicate_orders(orders)

    product_ids = set(products['product_id'])

    reasons = []

    for i, row in unique.iterrows():
        reason = validate_order(row.to_dict(), product_ids)
        reasons.append(reason)

    unique = unique.copy()
    unique['reason'] = reasons

    valid = unique[unique['reason'] == 'OK'].copy()
    rejected = unique[unique['reason'] != 'OK'].copy()

    duplicates = duplicates.copy()
    duplicates['reason'] = 'duplicate'

    rejected = pd.concat([rejected, duplicates], ignore_index=True)

    valid = valid.drop(columns='reason')
    valid = transform_orders(valid, products)

    return valid, rejected



In [ ]:

data = extract(RAW_DIR)

products = clean_products(data['products'])

valid, rejected = split_orders(data['orders'], products)

print('Ispravne:', len(valid))
print('Odbijene:', len(rejected))
print('Duplikati:', len(rejected[rejected['reason'] == 'duplicate']))
print('Broj kolona:', valid.shape[1])


In [ ]:
# 13. Tабела лиге. Написати функцију league_table(results: pd.DataFrame) -> pd.DataFrame која од резултата из задатка 12 прави табелу


def league_table(results: pd.DataFrame) -> pd.DataFrame:
    df = results.copy()

    df['won'] = (df['result'] == 'W').astype(int)
    df['drawn'] = (df['result'] == 'D').astype(int)
    df['lost'] = (df['result'] == 'L').astype(int)

    table = df.groupby('team_code')[['won', 'drawn', 'lost', 'goals_for', 'goals_against', 'points']].sum().reset_index()

    table['played'] = table['won'] + table['drawn'] + table['lost']
    table['goal_diff'] = table['goals_for'] - table['goals_against']

    table = table.sort_values(
        ['points', 'goal_diff', 'goals_for'],
        ascending=[False, False, True]
    )

    table = table.reset_index(drop=True)
    table['position'] = table.index + 1

    table = table[['position', 'team_code', 'played', 'won', 'drawn', 'lost', 'goals_for', 'goals_against', 'goal_diff', 'points']]

    return table


In [ ]:
table = league_table(results)

print(table.shape)
print(table[['position', 'team_code', 'points']].head(2))
print(table[table['team_code'].isin(['HGA', 'OKT'])][['team_code', 'played']])


In [ ]:
# 14 Прозори после утакмица. Написати функцију match_windows(results: pd.DataFrame) ->pd.DataFrame која за сваки ред из задатка 12 прави 4 реда: дан утакмице и три дана послење, са резултатом те утакмице


def match_windows(results: pd.DataFrame) -> pd.DataFrame:
    lista = []

    for d in range(4):
        df = results[['team_code', 'match_date', 'result']].copy()

        df['date'] = pd.to_datetime(df['match_date']) + pd.Timedelta(days=d)

        lista.append(df[['team_code', 'date', 'result']])

    windows = pd.concat(lista, ignore_index=True)

    windows = windows.rename(columns={'result': 'window'})

    return windows


In [ ]:

test = pd.DataFrame({
    'team_code': ['BRU'],
    'match_date': ['2026-08-15'],
    'result': ['W']
})

print(match_windows(test))


In [ ]:
# 15. Промет по колони. Написати функцију revenue_by(df: pd.DataFrame, column: str) ->pd.DataFrame


def revenue_by(df: pd.DataFrame, column: str) -> pd.DataFrame:
    rezultat = df.groupby(column)['total'].sum().reset_index(name='revenue')

    rezultat = rezultat.sort_values('revenue', ascending=False)

    return rezultat


In [ ]:

timovi = revenue_by(valid.dropna(subset=['team_code']), 'team_code')
kategorije = revenue_by(valid, 'category')

print(timovi.shape)
print(timovi.head(1))
print(timovi.tail(1))
print(kategorije.head(1))


In [ ]:
# 17 Load. Написати функцију load(outputs: dict, processed_dir: str) -> dict која сваки DataFrame из речника {име_фајла: DataFrame} уписује у processed_dirи враћа {име_фајла: број_редова}


def load(outputs: dict, processed_dir: str) -> dict:
    written = {}

    for name, df in outputs.items():
        path = os.path.join(processed_dir, name)

        if name.endswith('.parquet'):
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)

        written[name] = len(df)

    return written


In [ ]:

test = {'products.csv': products}

rezultat = load(test, PROCESSED_DIR)

print(rezultat)
